In [12]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("receipts.db")

df = pd.read_sql("SELECT * FROM receipts", conn)
df.head()

,id,company,date,total,address,confidence_json,raw_json
0,1,"HOTOR ( SUNGAT RENGIT ) SON, BHD.",23-01-2019,20.00,"Jalan Kurau, Sungai Rengit, 81620 Pengerang, J...","{""COMPANY"": 0.9355188267571586, ""ADDRESS"": 0.9...","{""COMPANY"": ""HOTOR ( SUNGAT RENGIT ) SON, BHD...."
1,2,PERNIAGAAN ZHENG,09/02/2078,436.20,JALAN PERMAS 9/6 BANDAR BARU PERMAS JAYA 81780...,"{""COMPANY"": 0.9209394752979279, ""ADDRESS"": 0.9...","{""COMPANY"": ""PERNIAGAAN ZHENG"", ""ADDRESS"": ""JA..."
2,3,CROSS CHANNEL SDN.,31/12/2017,7.95,"MERANTI 1, SEK. 3, BANDAR UTAMA BATANG KALI, 4...","{""COMPANY"": 0.9958662788073221, ""ADDRESS"": 0.9...","{""COMPANY"": ""CROSS CHANNEL SDN."", ""ADDRESS"": ""..."
3,4,SWC ENTERPRISE SDN BHD,06/01/2018,8.00,"Jalan Mahagoni 7/1, Sekysen 4, Bandar Utama, 4...","{""COMPANY"": 0.9968835711479187, ""ADDRESS"": 0.9...","{""COMPANY"": ""SWC ENTERPRISE SDN BHD"", ""ADDRESS..."
4,5,HORE MASTER HARDWARE & ELECTRICAL,22/12/2017,15.90,"& 115G, JALAN SETIA GEMEILANG SETIA ALAM, 4077...","{""COMPANY"": 0.9948287963867187, ""ADDRESS"": 0.9...","{""COMPANY"": ""HORE MASTER HARDWARE & ELECTRICAL..."


### Converting Rows to Documents

In [13]:
def row_to_text(row):
    return f"""
    Receipt:
    - Company: {row['company']}
    - Date: {row['date']}
    - Total: {row['total']} KES
    - Address: {row['address']}
    """
    
documents = df.apply(row_to_text, axis=1).tolist()

documents[:2]

['\n    Receipt:\n    - Company: HOTOR ( SUNGAT RENGIT ) SON, BHD.\n    - Date: 23-01-2019\n    - Total: 20.0 KES\n    - Address: Jalan Kurau, Sungai Rengit, 81620 Pengerang, Johor. el;\n    ',
 '\n    Receipt:\n    - Company: PERNIAGAAN ZHENG\n    - Date: 09/02/2078\n    - Total: 436.2 KES\n    - Address: JALAN PERMAS 9/6 BANDAR BARU PERMAS JAYA 81780 JOHOR TEL\n    ']

### Creating Embeddings using sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents, show_progress_bar=True)

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

In [16]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
EMBEDDINGS_PATH = BASE_DIR / "artifacts" / "doc_embeddings.npy"

np.save(EMBEDDINGS_PATH, doc_embeddings)

# doc_embeddings = np.load("doc_embeddings.npy")

### Storing in Vector DB (FAISS)

In [17]:
import faiss
import numpy as np

dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

In [39]:
INDEX_PATH = BASE_DIR / "artifacts" / "faiss.index"

faiss.write_index(index, str(INDEX_PATH))

In [18]:
# Retrieval Function

def retrieve(query, k=3):
    query_embedding = embed_model.encode([query])
    
    distances, indices = index.search(query_embedding, k)
    
    results = [documents[i] for i in indices[0]]
    
    return results

In [19]:
import os
from groq import Groq

from dotenv import load_dotenv
load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [30]:
# RAG Prompt

def build_prompt(query, retrieved_docs):
    context = "\n\n".join(retrieved_docs)

    return f"""
You are a financial assistant that answers questions about receipts.
Use ONLY the context below. Do NOT invent information.

Context (each receipt):
{context}

Fields you can use:
- company
- date
- total
- address

Question:
{query}

Instructions:
- Only use the receipts in the context.

Answer clearly and concisely.
"""

In [31]:
# Ask function (full RAG pipeline)
def ask_llm(query):
    docs = retrieve(query, k=3)
    
    prompt = build_prompt(query, docs)
    
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [35]:
ask_llm("Which company has the most total amount of money spent at?")

'BECON STATIONER Becon Enterprise Sdn Bhd with a total of 270.1 KES.'

In [ ]:
pd.read_sql("SELECT * FROM receipts WHERE company==", conn)

In [26]:
pd.read_sql("""
SELECT company, SUM(total) AS total_spent
FROM receipts
GROUP BY company
ORDER BY total_spent DESC
""", conn)

,company,total_spent
0,None,1672.195
1,PERNIAGAAN ZHENG,872.400
2,SYARIKAT PERNIAGAAN GIN KEE,639.070
3,PINGHWAI TRADING SDN BHD,616.590
4,KEDAI PAPAN YEW CHUAN,374.710
...,...,...
76,IMAGE (M) SDN BHD,NaN
77,EVERGREEN LIGHT SDN BHD,NaN
78,CUPER.SEVEN CASH 6 CARRY SEN SHO,NaN
79,AFON CO. (M) BHD,NaN


In [33]:
ask_llm("Show me receipts from December")

'There are no receipts from December.'

### Rebuilding index from DB

In [42]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import sqlite3

# Load DB
conn = sqlite3.connect("receipts.db")
df = pd.read_sql("SELECT * FROM receipts", conn)

# Convert to text
def row_to_text(row):
    return f"""
    Company: {row['company']}
    Date: {row['date']}
    Total: {row['total']} KES
    Address: {row['address']}
    """

documents = df.apply(row_to_text, axis=1).tolist()

# Embed
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(documents)

# Build FAISS
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

# Save
INDEX_PATH = BASE_DIR / "artifacts" / "faiss.index"
EMBEDDINGS_PATH = BASE_DIR / "artifacts" / "doc_embeddings.npy"

faiss.write_index(index, str(INDEX_PATH))
np.save(EMBEDDINGS_PATH, embeddings)